In [11]:
import pandas as pd
import json

In [12]:
file_path = '../data/miband/20260406_8278074507_MiFitness_ams1_data_copy/20260406_8278074507_MiFitness_hlth_center_aggregated_fitness_data.csv'
df_raw = pd.read_csv(file_path)

In [13]:
df_spo2 = df_raw[(df_raw['Key'] == 'spo2') & (df_raw['Tag'] == 'daily_report')].copy()

spo2_data_list = [json.loads(val) for val in df_spo2['Value']]
df_spo2_normalized = pd.json_normalize(spo2_data_list)

df_spo2_result = df_spo2[['Uid', 'Sid', 'Time']].reset_index(drop=True).join(df_spo2_normalized)

df_spo2_result = df_spo2_result.drop(['lack_spo2_count', 'latest_spo2.spo2', 'latest_spo2.time'], axis=1)

df_spo2_result

,Uid,Sid,Time,avg_spo2,min_spo2,max_spo2
0,8278074507,default,1736812800,98,98,98
1,8278074507,default,1737763200,98,98,98
2,8278074507,default,1738108800,96,92,98
3,8278074507,default,1738195200,97,91,99
4,8278074507,default,1738281600,97,94,99
...,...,...,...,...,...,...
281,8278074507,default,1775088000,97,95,99
282,8278074507,default,1775174400,97,93,99
283,8278074507,default,1775260800,98,95,99
284,8278074507,default,1775347200,98,94,99


In [14]:
df_spo2_result['Date'] = pd.to_datetime(df_spo2_result['Time'], unit='s')

df_spo2_result = df_spo2_result.drop(columns=['Time', 'Uid', 'Sid'])

df_spo2_result

,avg_spo2,min_spo2,max_spo2,Date
0,98,98,98,2025-01-14
1,98,98,98,2025-01-25
2,96,92,98,2025-01-29
3,97,91,99,2025-01-30
4,97,94,99,2025-01-31
...,...,...,...,...
281,97,95,99,2026-04-02
282,97,93,99,2026-04-03
283,98,95,99,2026-04-04
284,98,94,99,2026-04-05


In [15]:
import os

output_dir = '../data/processed_data'
output_path = os.path.join(output_dir, 'miband_spo2_processed.csv')
df_spo2_result.to_csv(output_path, index=False)